In [1]:
"""
Two methods for comparing stations and observation:
    1. Smooth JJA tasmin/max/avg simulated fields, grab closest station
    2. Use multivariate regression sfc_scheme (MVRM) trained on station data to reproduce the simulation

Load seasonal averages from station data, and modelled seasonal averages at station points
"""

import xarray as xr
import numpy as np
import plotly.graph_objects as go
from Montreal_UHI_toolbox import add_field_to_stations, add_blurred_field_to_stations, obs

/runoff/gulley/.miniconda3/lib/python3.12/site-packages/gribapi/__init__.py:23: UserWarning: ecCodes 2.39.0 or higher is recommended. You are running version 2.14.1
  warnings.warn(


In [13]:
station_set = obs

# Load all season data from observation and otherwise into station_set dataset
seasonal = {}
seasonal_std = {}
path = '/runoff/gulley/St_Laurent/intermediates'
for season in ['SON','DJF','MAM','JJA']:
    for field in ['tasmin','tasmax','tasavg']:

        # Loading simulated season averages 
        seasonal[f'{field}_C'] = xr.open_zarr(f'{path}/sim/seasonal/seasons_avg_{field}_noTEB.zarr')[field].sel(season=season) - 273.15
        seasonal[f'{field}_T'] = xr.open_zarr(f'{path}/sim/seasonal/seasons_avg_{field}_TEB.zarr')[field].sel(season=season) - 273.15

        # Loading simulated season standard deviations
        seasonal_std[f'{field}_C'] = xr.open_zarr(f'{path}/sim/seasonal/seasons_std_{field}_noTEB.zarr')[field].sel(season=season)
        seasonal_std[f'{field}_T'] = xr.open_zarr(f'{path}/sim/seasonal/seasons_std_{field}_TEB.zarr')[field].sel(season=season)

        
        # Adding smoothed simulated season averages to nearest station
        station_set = add_blurred_field_to_stations(seasonal[f'{field}_C'],stations=station_set,name=f'{field}_{season}_avg_C')
        station_set = add_blurred_field_to_stations(seasonal[f'{field}_T'],stations=station_set,name=f'{field}_{season}_avg_T')

        # Adding smoothed season standard deviation to nearest station
        station_set = add_blurred_field_to_stations(seasonal_std[f'{field}_C'],stations=station_set,name=f'{field}_{season}_std_C')
        station_set = add_blurred_field_to_stations(seasonal_std[f'{field}_T'],stations=station_set,name=f'{field}_{season}_std_T')

    # Loading station season averages
    tasmax_S = xr.open_zarr(f'{path}/station/seasons_avg_tasmax.zarr')['tasmax'].sel(season=season)
    tasmin_S = xr.open_zarr(f'{path}/station/seasons_avg_tasmin.zarr')['tasmin'].sel(season=season)
    tasavg_S = xr.open_zarr(f'{path}/station/seasons_avg_tas.zarr')['tas'].sel(season=season).rename('tasavg')
    # Loading station season standard deviations
    tasmax_std_S = xr.open_zarr(f'{path}/station/seasons_std_tasmax.zarr')['tasmax'].sel(season=season)
    tasmin_std_S = xr.open_zarr(f'{path}/station/seasons_std_tasmin.zarr')['tasmin'].sel(season=season)
    tasavg_std_S = xr.open_zarr(f'{path}/station/seasons_std_tas.zarr')['tas'].sel(season=season).rename('tasavg')


    for da in [tasmax_S,tasmin_S,tasavg_S]:
        station_set[f'{da.name}_{season}_avg_S'] = da

    for da in [tasmax_std_S,tasmin_std_S,tasavg_std_S]:
        station_set[f'{da.name}_{season}_std_S'] = da

    station_set = station_set.dropna(dim='station', subset=[f'tasmax_{season}_avg_S',f'tasmin_{season}_avg_S',f'tasmin_{season}_avg_S'])

In [14]:
sfc_schemes = ['C','T']
for season in ['SON','DJF','MAM','JJA']:
    for field in ['tasmin','tasmax','tasavg']:
        for sfc_scheme in sfc_schemes:
            # Subtract observed values from modelled values of season averages
            station_set[f'{field}_{season}_avg_obsDiff_{sfc_scheme}'] = station_set[f'{field}_{season}_avg_{sfc_scheme}'] - station_set[f'{field}_{season}_avg_S']

            # Differences in seasonal variability
            station_set[f'{field}_{season}_std_obsDiff_{sfc_scheme}'] = station_set[f'{field}_{season}_std_{sfc_scheme}'] - station_set[f'{field}_{season}_std_S']
df = station_set

In [31]:
# field_name = 'Minimum Daily Temperature'
# field = 'tasmin'
for season in ['SON','DJF','MAM','JJA']:
    for field,field_name in zip(['tasmin','tasmax','tasavg'],
                    [f'<sup>{season}</sup><SPAN STYLE="text-decoration:overline">T</SPAN><sub>min</sub>',f'<sup>{season}</sup><SPAN STYLE="text-decoration:overline">T</SPAN><sub>max</sub>',f'<sup>{season}</sup><SPAN STYLE="text-decoration:overline">T</SPAN><sub>avg</sub>']):
    # for field,field_name in zip(['tasmin','tasmax'],
    #                 [f'<sup>{season}</sup><SPAN STYLE="text-decoration:overline">T</SPAN><sub>min</sub>','<SPAN STYLE="text-decoration:overline">T</SPAN><sub>max</sub>']):
        fig = go.Figure()
        diff_C = df[f'{field}_{season}_avg_C'].values - df[f'{field}_{season}_avg_S'].values
        diff_T = df[f'{field}_{season}_avg_T'].values - df[f'{field}_{season}_avg_S'].values
        
        # Draw vertical lines showing differences at each station
        for i in range(len(df['station_name'])):

            # Adding lines to show the differences from observation
            fig.add_trace(go.Scatter(
                x=[df[f'{field}_{season}_avg_S'][i], df[f'{field}_{season}_avg_S'][i]],
                y=[df[f'{field}_{season}_avg_S'][i], df[f'{field}_{season}_avg_T'][i]],
                mode='lines',
                line=dict(color='grey', width=0.5),
                showlegend=False,
                legendgroup='TEB+CLASS',
                legendgrouptitle={'text': 'TEB+CLASS'},
                hoverinfo='skip'
            ))
            fig.add_trace(go.Scatter(
                x=[df[f'{field}_{season}_avg_S'][i], df[f'{field}_{season}_avg_S'][i]],
                y=[df[f'{field}_{season}_avg_S'][i], df[f'{field}_{season}_avg_C'][i]],
                mode='lines',
                line=dict(color='grey', width=0.5),
                showlegend=False,
                legendgroup='CLASS',
                legendgrouptitle={'text': 'CLASS'},
                hoverinfo='skip'
            ))
        
        # Scatter station markers (observation vs observation)
        fig.add_trace(go.Scatter(
                x=df[f'{field}_{season}_avg_S'].values,
                y=df[f'{field}_{season}_avg_S'].values,
                mode='markers',
                hovertext=df['station_name'].values,
                hoverinfo='text+x+y',
                marker=dict(color='black'),
                hovertemplate=(
                'Station: %{customdata}<br>'
                'Observation: %{x:.2f}°C<br>'
                ),
                customdata=df['station_name'].values,
                legendgroup='Observation',
                legendgrouptitle={'text': 'Observation'},
                name=''
            )
        )
        
        # Scatter CLASS-determined station temperatures (CLASS vs observation)
        fig.add_trace(go.Scatter(
                x=df[f'{field}_{season}_avg_S'].values,
                y=df[f'{field}_{season}_avg_C'].values,
                mode='markers',
                hovertext=df['station_name'].values,
                hoverinfo='text+x+y',
                marker=dict(color='blue'),
                hovertemplate=(
                'Station: %{customdata[0]}<br>'
                'Model (CLASS): %{y:.2f}°C<br>'
                'Observation: %{x:.2f}°C<br>'
                'ΔT: %{customdata[1]:+.2f}°C<extra></extra>'
                ),
                customdata=np.stack([df['station_name'].values, diff_C], axis=-1),
                legendgroup='CLASS',
                legendgrouptitle={'text': 'CLASS'},
                name=''
            )
        )
        
        # Scatter TEB+CLASS-determined station temperatures (TEB+CLASS vs observation)
        fig.add_trace(go.Scatter(
                x=df[f'{field}_{season}_avg_S'].values,
                y=df[f'{field}_{season}_avg_T'].values,
                mode='markers',
                hovertext=df['station_name'].values,
                hoverinfo='text+x+y',
                marker=dict(color='red'),
                hovertemplate=(
                'Station: %{customdata[0]}<br>'
                'Model (TEB+CLASS): %{y:.2f}°C<br>'
                'Observation: %{x:.2f}°C<br>'
                'ΔT: %{customdata[1]:+.2f}°C<extra></extra>'
                ),
                customdata=np.stack([df['station_name'].values, diff_T], axis=-1),
                legendgroup='TEB+CLASS',
                legendgrouptitle={'text': 'TEB+CLASS'},
                name=''
            )
        )
        
        # For setting the bounds of the graph 1:1 scale and aspect ratio
        max_xy = max([max(df[f'{field}_{season}_avg_C'].values),max(df[f'{field}_{season}_avg_T'].values),max(df[f'{field}_{season}_avg_S'].values)]) + 1.0
        min_xy = min([min(df[f'{field}_{season}_avg_C'].values),min(df[f'{field}_{season}_avg_T'].values),min(df[f'{field}_{season}_avg_S'].values)]) - 1.0

        # fig.update_xaxes(range=[min_xy,max_xy])
        # fig.update_yaxes(range=[min_xy,max_xy])
        # Or alternatively:
        midpoint = (min_xy + max_xy)/2
        window = 3
        min_xy = midpoint - window
        max_xy = midpoint + window
        fig.update_xaxes(range=[min_xy,max_xy])
        fig.update_yaxes(range=[min_xy,max_xy])
        

        # Draw observation line
        fig.add_trace(go.Scatter(
                x=[min_xy,max_xy],
                y=[min_xy,max_xy],
                mode='lines',
                line=dict(color='black', width=0.5),
                showlegend=False,
                hoverinfo='skip'
            ))

        # Titling and layout
        fig.update_layout(
            xaxis_title=f'Observed {field_name} (°C)',
            yaxis_title=f'Modelled {field_name} (°C)',
            title=f'Modelled vs Observed {field_name}',
            width = 900,height=900
        )
        # fig.show()
        fig.write_html(f'/home/gulley/UHI_HW_MTL/info/plots/accuracy_{season}_{field}.html')